# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
load_dotenv()

## Defining Variables

### Defining Constants

In [0]:
CONTAINER_NAME= os.getenv('CONTAINER_NAME')
STORAGE_ACCOUNT_NAME= os.getenv('STORAGE_ACCOUNT_NAME')
EXTERNAL_LOCATION = f'abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/'
FILE_LIST = dbutils.fs.ls(EXTERNAL_LOCATION)
EXTERNAL_FILES = [file.name for file in FILE_LIST]
BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

### Defining Table Location

In [0]:
country_master_path=EXTERNAL_LOCATION+EXTERNAL_FILES[0]
customer_path=EXTERNAL_LOCATION+EXTERNAL_FILES[1]
employee_path=EXTERNAL_LOCATION+EXTERNAL_FILES[2]
fx_rate_path=EXTERNAL_LOCATION+EXTERNAL_FILES[3]
opportunity_path=EXTERNAL_LOCATION+EXTERNAL_FILES[4]
product_path=EXTERNAL_LOCATION+EXTERNAL_FILES[5]

## Creating Spark Dataframes

In [0]:
country_master_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header", True)\
      .option("inferSchema", True)\
      .load(country_master_path))

customer_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header", True)\
      .option("inferSchema", True)\
      .load(customer_path))

employee_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header", True)\
      .option("inferSchema", True)\
      .load(employee_path))

fx_rate_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header", True)\
      .option("inferSchema", True)\
      .load(fx_rate_path))

opportunity_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header", True)\
      .option("inferSchema",True)\
      .load(opportunity_path))

product_df = (spark.read\
      .format("csv")\
      .option("mode", "FAILFAST")\
      .option("header",True)\
      .option("inferSchema",True)\
      .load(product_path))

## Changing Column Names for Ingestion

In [0]:
import re
from pyspark.sql import DataFrame

def to_snake_case(col_name: str) -> str:
    # Replace spaces and special chars with underscore
    col_name = re.sub(r'[^a-zA-Z0-9]', '_', col_name)
    
    # Convert CamelCase to snake_case
    col_name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    col_name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', col_name)
    
    # Convert to lowercase
    col_name = col_name.lower()
    
    # Remove multiple underscores
    col_name = re.sub('_+', '_', col_name)
    
    # Remove leading/trailing underscores
    col_name = col_name.strip('_')
    
    return col_name

def rename_columns_snake_case(df: DataFrame) -> DataFrame:
    new_cols = [to_snake_case(c) for c in df.columns]
    return df.toDF(*new_cols)

In [0]:
product_df = rename_columns_snake_case(product_df)
customer_df = rename_columns_snake_case(customer_df)
employee_df = rename_columns_snake_case(employee_df)
opportunity_df = rename_columns_snake_case(opportunity_df)
fx_rate_df = rename_columns_snake_case(fx_rate_df)
country_master_df=rename_columns_snake_case(country_master_df)

## Saving Dataframe

In [0]:

BRONZE_SCHEMA_PATH=os.getenv('BRONZE_SCHEMA_PATH')

### Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {BRONZE_SCHEMA_PATH}""")

In [0]:
country_master_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_country_master`")
customer_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_customer`")
employee_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_employee`")
fx_rate_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_fx_rate`")
opportunity_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_opportunity`")
product_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA_PATH}.`bronze_product`")   